In [ ]:
# ============================================================
# ROUNDOFF NOISE IN FIR AND IIR FILTERS
# ============================================================
#
# This notebook compares fixed-point roundoff-noise propagation
# in FIR and IIR digital filters.
#
# The central distinction is:
#
#       FIR:
#           finite feedforward accumulation of quantization noise
#
#       IIR:
#           quantization noise injected into a feedback structure
#           and amplified by the recursive dynamics.
#
#
# ============================================================
# QUANTIZATION-NOISE MODEL
# ============================================================
#
# With K fractional bits,
#
#                   Delta = 2^(-K)
#
# and, under the standard PQN rounding-noise model,
#
#                   sigma_q^2 = Delta^2 / 12.
#
# The individual PQN sources are assumed to be:
#
#       - zero mean,
#       - uniformly distributed,
#       - mutually uncorrelated,
#       - sufficiently independent of the relevant signals.
#
#
# ============================================================
# FIR FILTER
# ============================================================
#
# A length-N FIR filter is described by
#
#       y[n] = sum_{k=0}^{N-1} h[k] x[n-k].
#
# The supplied theoretical section gives the output-noise power
#
#       P_FIR
#
#       = (Delta^2 / 12)
#         [ sum_{n=1}^{N-1} h^2[n] + (N+1) ].
#
# The two terms have different origins:
#
#       sum h^2[n]
#
# represents the input quantization noise propagated through
# the FIR filter, whereas
#
#       N + 1
#
# represents the contribution of the product/output PQN sources.
#
#
# ============================================================
# ILLUSTRATIVE FIR USED IN THIS NOTEBOOK
# ============================================================
#
# To make the FIR relation interactive without introducing a
# separate filter-design problem, this notebook uses a normalized
# moving-average FIR:
#
#                   h[n] = 1/N,
#                   0 <= n < N.
#
# Therefore,
#
#       sum_{n=1}^{N-1} h^2[n]
#
#       = (N-1) / N^2.
#
# This is a deliberate pedagogical choice. The general FIR
# relation remains valid for arbitrary coefficient sets h[n].
#
#
# ============================================================
# FIRST-ORDER IIR FILTER
# ============================================================
#
# The first-order recursive system considered in the section has
#
#                   H(z) = 1 / (1 - alpha z^(-1))
#
# and
#
#                   h[n] = alpha^n u[n].
#
# Hence,
#
#       sum_{n=0}^{infinity} h^2[n]
#
#       = 1 / (1 - alpha^2),
#
# for
#
#                   |alpha| < 1.
#
# With two uncorrelated PQN sources, each having variance
#
#                   Delta^2 / 12,
#
# the output-noise power becomes
#
#       P_IIR
#
#       = 2 (Delta^2 / 12)
#         [1 / (1 - alpha^2)]
#
#       = Delta^2 / [6(1 - alpha^2)].
#
#
# ============================================================
# HOW TO USE THIS NOTEBOOK
# ============================================================
#
# 1. FIR OUTPUT NOISE VS N
#
#    Active controls:
#
#       K
#       FIR length N
#
#    Observe how the FIR output-noise power changes as the number
#    of taps increases.
#
#
# 2. FIR NOISE CONTRIBUTIONS
#
#    Active controls:
#
#       K
#       FIR length N
#
#    The graph separates:
#
#       - propagated input-quantization noise,
#       - internal/output PQN contribution,
#       - total FIR noise power.
#
#
# 3. IIR OUTPUT NOISE VS ALPHA
#
#    Active controls:
#
#       K
#       alpha
#
#    Observe the noise amplification produced by the feedback
#    pole as |alpha| approaches 1.
#
#
# 4. FIR VS IIR
#
#    Active controls:
#
#       K
#       FIR length N
#       alpha
#
#    The selected FIR noise power is shown together with the
#    first-order IIR noise-power curve.
#
#
# 5. EFFECT OF WORD LENGTH K
#
#    Active controls:
#
#       FIR length N
#       alpha
#
#    Both FIR and IIR output-noise powers are plotted as functions
#    of K.
#
#
# ============================================================
# WHAT WE EXPECT TO OBSERVE
# ============================================================
#
# FIR FILTER
#
# An FIR structure contains no feedback loop.
#
# Its noise contributions are accumulated through a finite
# feedforward structure. Increasing the number of quantizers
# increases the internal-noise contribution, but there is no
# recursive amplification mechanism.
#
#
# IIR FILTER
#
# The IIR noise power contains the factor
#
#                   1 / (1 - alpha^2).
#
# Therefore, as
#
#                   |alpha| -> 1,
#
# the output-noise power increases very rapidly.
#
# This is the characteristic effect of feedback on roundoff
# noise.
#
#
# FIR VS IIR
#
# The FIR noise remains finite for any finite filter length.
#
# The first-order IIR noise can become very large as the pole
# approaches the unit circle.
#
# Thus, the main difference is not simply that both filters
# contain quantizers, but that the IIR structure can repeatedly
# propagate and amplify internally generated noise.
#
#
# WORD LENGTH
#
# Since
#
#                   Delta = 2^(-K),
#
# all PQN variances contain the factor
#
#                   Delta^2 = 2^(-2K).
#
# Therefore, increasing K by one bit reduces the basic
# quantization-noise variance by a factor of four.
#
#
# ============================================================
# IMPORTANT INTERPRETATION
# ============================================================
#
# This notebook visualizes the analytical PQN relations given in
# the theoretical development.
#
# The PQN replacement is a statistical model. Its validity
# requires the corresponding quantization-theorem assumptions to
# be satisfied at least approximately.
#
# The comparison is therefore about noise behavior UNDER THE PQN
# MODEL, not a claim that every real quantizer always produces
# perfectly white independent noise.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive
from IPython.display import display


# ------------------------------------------------------------
# Basic quantization functions
# ------------------------------------------------------------

def quantization_step(K):

    return 2.0**(-K)


def one_source_variance(K):

    Delta = quantization_step(K)

    return Delta**2 / 12.0


# ------------------------------------------------------------
# Illustrative FIR coefficients
# ------------------------------------------------------------

def moving_average_coefficients(N):

    return np.ones(N) / N


# ------------------------------------------------------------
# FIR noise terms
# ------------------------------------------------------------

def fir_input_noise_contribution(N, K):

    h = moving_average_coefficients(N)

    sigma_q2 = one_source_variance(K)

    coefficient_energy_from_1 = np.sum(h[1:]**2)

    return sigma_q2 * coefficient_energy_from_1


def fir_internal_noise_contribution(N, K):

    sigma_q2 = one_source_variance(K)

    return sigma_q2 * (N + 1)


def fir_total_noise_power(N, K):

    return fir_input_noise_contribution(N, K) + fir_internal_noise_contribution(N, K)


# ------------------------------------------------------------
# IIR noise power
# ------------------------------------------------------------

def iir_noise_gain(alpha):

    return 1.0 / (1.0 - alpha**2)


def iir_total_noise_power(alpha, K):

    Delta = quantization_step(K)

    return (Delta**2 / 6.0) / (1.0 - alpha**2)


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.fi-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.fi-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.fi-howto {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7cfe0;
    border-left: 6px solid #667da8;
    background: #f8f9fc;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.fi-howto-title {
    font-size: 13px;
    font-weight: bold;
    color: #40587d;
    margin-bottom: 5px;
}

.fi-observe {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7d8c9;
    border-left: 6px solid #3c8a4e;
    background: #f7fbf7;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.fi-observe-title {
    font-size: 13px;
    font-weight: bold;
    color: #245c31;
    margin-bottom: 5px;
}

.fi-model {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #d5c58a;
    border-left: 6px solid #b8860b;
    background: #fffaf0;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.fi-model-title {
    font-size: 13px;
    font-weight: bold;
    color: #8a6500;
    margin-bottom: 5px;
}

.fi-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.fi-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.fi-info {
    font-size: 13px;
    line-height: 1.52;
}

.fi-label {
    display: inline-block;
    min-width: 250px;
    font-weight: bold;
}

.fi-value {
    font-size: 14px;
    font-weight: bold;
}

.fi-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 6px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="fi-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Roundoff Noise in FIR and IIR Filters
    </div>

</div>
""")


# ------------------------------------------------------------
# Visible introductory documentation
# ------------------------------------------------------------

description_html = HTML("""
<div class="fi-root">

    <div class="fi-description">

        <b>What this notebook demonstrates</b><br><br>

        In fixed-point digital filters, arithmetic quantizers can be replaced
        under the PQN approximation by additive noise sources with

        <div style="text-align:center; margin:6px 0;">
            <b>
            Δ = 2<sup>−K</sup>,
            &nbsp;&nbsp;
            σ²<sub>q</sub> = Δ²/12.
            </b>
        </div>

        The important structural difference is that an <b>FIR filter</b>
        contains only a finite feedforward path, whereas an <b>IIR filter</b>
        contains feedback. Consequently, internally generated noise can be
        repeatedly propagated and amplified in the IIR case.

    </div>


    <div class="fi-howto">

        <div class="fi-howto-title">
            How to use this notebook
        </div>

        <b>FIR output noise vs N:</b>
        vary K and the FIR length N and observe the total FIR output-noise
        power.<br><br>

        <b>FIR noise contributions:</b>
        inspect separately the noise propagated through the FIR coefficients
        and the noise produced by the internal/output PQN sources.<br><br>

        <b>IIR output noise vs alpha:</b>
        vary α and K and observe the factor
        <b>1/(1−α²)</b> amplifying the noise as |α| approaches one.<br><br>

        <b>FIR vs IIR:</b>
        choose N and K for the FIR filter and compare its noise power with
        the first-order IIR noise-power curve. The selected value of α is
        also marked on the IIR curve.<br><br>

        <b>Effect of word length K:</b>
        choose N and α and compare how FIR and IIR noise powers decrease as
        the fractional word length increases.<br><br>

        Controls that do not affect the currently selected display are
        automatically disabled.

    </div>


    <div class="fi-observe">

        <div class="fi-observe-title">
            What we expect to observe
        </div>

        <b>FIR filter:</b>
        noise contributions are accumulated through a finite feedforward
        structure. There is no recursive noise-amplification mechanism.<br><br>

        <b>IIR filter:</b>
        the noise power contains the factor
        <b>1/(1−α²)</b>. As |α| approaches one, this factor becomes very
        large and the output-noise power rises sharply.<br><br>

        <b>FIR versus IIR:</b>
        the FIR result remains finite for finite N, whereas the IIR result
        can become very large when its pole approaches the unit circle.
        This demonstrates the fundamental effect of feedback on internally
        generated roundoff noise.<br><br>

        <b>Word length:</b>
        increasing K decreases Δ and therefore reduces all PQN powers.
        Because the power contains Δ² = 2<sup>−2K</sup>, one additional bit
        reduces the basic quantization-noise variance by a factor of four.

    </div>


    <div class="fi-model">

        <div class="fi-model-title">
            FIR model used for the demonstration
        </div>

        The general FIR relation depends on the actual coefficients h[n].
        To keep this notebook focused on roundoff noise rather than filter
        design, the FIR example uses a normalized moving-average filter:

        <div style="text-align:center; margin:6px 0;">
            <b>
            h[n] = 1/N,
            &nbsp;&nbsp;
            0 ≤ n &lt; N.
            </b>
        </div>

        The FIR output-noise relation implemented here is the relation from
        the theoretical section:

        <div style="text-align:center; margin:6px 0;">
            <b>
            P<sub>FIR</sub>
            =
            (Δ²/12)
            [Σ<sub>n=1</sub><sup>N−1</sup>h²[n] + (N+1)].
            </b>
        </div>

        For the first-order IIR example:

        <div style="text-align:center; margin:6px 0;">
            <b>
            P<sub>IIR</sub>
            =
            Δ² / [6(1−α²)],
            &nbsp;&nbsp;
            |α| &lt; 1.
            </b>
        </div>

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic numerical summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(
    width='610px',
    min_width='610px',
    overflow='visible'
)


# ------------------------------------------------------------
# Main interactive function
# ------------------------------------------------------------

def plot_fir_iir_noise(display_mode='FIR output noise vs N', K=8, N=12, alpha=0.85):


    # --------------------------------------------------------
    # Current basic values
    # --------------------------------------------------------

    Delta = quantization_step(K)

    sigma_q2 = one_source_variance(K)


    # --------------------------------------------------------
    # Current FIR quantities
    # --------------------------------------------------------

    h = moving_average_coefficients(N)

    fir_energy_term = np.sum(h[1:]**2)

    fir_input_component = fir_input_noise_contribution(N, K)

    fir_internal_component = fir_internal_noise_contribution(N, K)

    fir_total = fir_total_noise_power(N, K)


    # --------------------------------------------------------
    # Current IIR quantities
    # --------------------------------------------------------

    iir_gain = iir_noise_gain(alpha)

    iir_total = iir_total_noise_power(alpha, K)


    # --------------------------------------------------------
    # Dynamic summary
    # --------------------------------------------------------

    summary_html.value = f"""
    <div class="fi-box">

        <div class="fi-title">
            Current Roundoff-Noise Data
        </div>

        <div class="fi-info">

            <span class="fi-label">Selected display</span>
            {display_mode}
            <br>

            <span class="fi-label">Fractional bits</span>
            K = <span class="fi-value">{K}</span>
            <br>

            <span class="fi-label">Quantization step</span>
            Δ = {Delta:.8e}
            <br>

            <span class="fi-label">One PQN-source variance</span>
            Δ²/12 = {sigma_q2:.8e}
            <br><br>

            <b>FIR moving-average example</b>
            <br>

            <span class="fi-label">FIR length</span>
            N = <span class="fi-value">{N}</span>
            <br>

            <span class="fi-label">Σ h²[n], n=1,...,N−1</span>
            {fir_energy_term:.8e}
            <br>

            <span class="fi-label">Propagated input-noise power</span>
            {fir_input_component:.8e}
            <br>

            <span class="fi-label">Internal/output PQN power</span>
            {fir_internal_component:.8e}
            <br>

            <span class="fi-label">Total FIR noise power</span>
            <span class="fi-value">{fir_total:.8e}</span>
            <br><br>

            <b>First-order IIR example</b>
            <br>

            <span class="fi-label">Pole coefficient</span>
            α = <span class="fi-value">{alpha:.4f}</span>
            <br>

            <span class="fi-label">Noise gain</span>
            1/(1−α²) = {iir_gain:.8f}
            <br>

            <span class="fi-label">Total IIR noise power</span>
            <span class="fi-value">{iir_total:.8e}</span>

        </div>

        <div class="fi-note">
            The FIR example uses h[n] = 1/N. The IIR result assumes two
            mutually uncorrelated PQN sources and |α| &lt; 1.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # One large figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(11.8, 5.0)
    )


    # ========================================================
    # FIR OUTPUT NOISE VS N
    # ========================================================

    if display_mode == 'FIR output noise vs N':

        N_values = np.arange(
            2,
            65
        )


        powers = np.array(
            [
                fir_total_noise_power(n_value, K)
                for n_value in N_values
            ]
        )


        ax.plot(
            N_values,
            powers,
            linewidth=1.8,
            label='FIR total output-noise power'
        )


        ax.plot(
            N,
            fir_total,
            'o',
            markersize=9,
            label='Current FIR length'
        )


        ax.set_xlabel(
            'FIR length N'
        )


        ax.set_ylabel(
            'Output-noise power'
        )


        ax.set_title(
            f'FIR Roundoff-Noise Power versus Filter Length — K = {K}',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            frameon=False,
            fontsize=9
        )


    # ========================================================
    # FIR NOISE CONTRIBUTIONS
    # ========================================================

    elif display_mode == 'FIR noise contributions':

        positions = np.arange(
            3
        )


        values = [
            fir_input_component,
            fir_internal_component,
            fir_total
        ]


        labels = [
            'Input-noise\npropagation',
            'Internal/output\nPQN sources',
            'Total FIR\nnoise'
        ]


        ax.bar(
            positions,
            values,
            width=0.55
        )


        ax.set_xticks(
            positions
        )


        ax.set_xticklabels(
            labels
        )


        ax.set_ylabel(
            'Noise power'
        )


        ax.set_title(
            f'Components of FIR Roundoff Noise — N = {N}, K = {K}',
            fontsize=12
        )


        ax.grid(
            True,
            axis='y',
            linestyle=':',
            alpha=0.5
        )


        maximum_value = max(values)


        for position, value in zip(positions, values):

            ax.text(
                position,
                value + 0.025 * maximum_value,
                f'{value:.3e}',
                ha='center',
                va='bottom',
                fontsize=9
            )


        ax.set_ylim(
            0.0,
            1.18 * maximum_value
        )


    # ========================================================
    # IIR OUTPUT NOISE VS ALPHA
    # ========================================================

    elif display_mode == 'IIR output noise vs alpha':

        alpha_values = np.linspace(
            -0.98,
            0.98,
            800
        )


        powers = np.array(
            [
                iir_total_noise_power(alpha_value, K)
                for alpha_value in alpha_values
            ]
        )


        ax.plot(
            alpha_values,
            powers,
            linewidth=1.8,
            label='First-order IIR output-noise power'
        )


        ax.plot(
            alpha,
            iir_total,
            'o',
            markersize=9,
            label='Current α'
        )


        ax.axvline(
            alpha,
            linestyle='--',
            linewidth=1.0
        )


        ax.set_xlabel(
            'Pole coefficient α'
        )


        ax.set_ylabel(
            'Output-noise power'
        )


        ax.set_title(
            f'IIR Roundoff-Noise Amplification versus Pole Position — K = {K}',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            frameon=False,
            fontsize=9
        )


    # ========================================================
    # FIR VS IIR
    # ========================================================

    elif display_mode == 'FIR vs IIR':

        alpha_values = np.linspace(
            -0.98,
            0.98,
            800
        )


        iir_powers = np.array(
            [
                iir_total_noise_power(alpha_value, K)
                for alpha_value in alpha_values
            ]
        )


        fir_reference = np.full_like(
            alpha_values,
            fir_total
        )


        ax.plot(
            alpha_values,
            iir_powers,
            linewidth=1.9,
            label='First-order IIR'
        )


        ax.plot(
            alpha_values,
            fir_reference,
            '--',
            linewidth=1.8,
            label=f'FIR moving average, N = {N}'
        )


        ax.plot(
            alpha,
            iir_total,
            'o',
            markersize=9,
            label='Current IIR operating point'
        )


        ax.set_xlabel(
            'IIR pole coefficient α'
        )


        ax.set_ylabel(
            'Output-noise power'
        )


        ax.set_title(
            f'FIR versus IIR Roundoff Noise — K = {K}',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=3,
            frameon=False,
            fontsize=9
        )


    # ========================================================
    # EFFECT OF WORD LENGTH K
    # ========================================================

    else:

        K_values = np.arange(
            2,
            17
        )


        fir_powers = np.array(
            [
                fir_total_noise_power(N, k_value)
                for k_value in K_values
            ]
        )


        iir_powers = np.array(
            [
                iir_total_noise_power(alpha, k_value)
                for k_value in K_values
            ]
        )


        ax.plot(
            K_values,
            fir_powers,
            'o-',
            linewidth=1.8,
            markersize=5,
            label=f'FIR, N = {N}'
        )


        ax.plot(
            K_values,
            iir_powers,
            's--',
            linewidth=1.8,
            markersize=5,
            label=f'IIR, α = {alpha:.2f}'
        )


        ax.set_yscale(
            'log'
        )


        ax.set_xticks(
            K_values
        )


        ax.set_xlabel(
            'Fractional bits K'
        )


        ax.set_ylabel(
            'Output-noise power'
        )


        ax.set_title(
            'Effect of Fractional Word Length on FIR and IIR Roundoff Noise',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            frameon=False,
            fontsize=9
        )


    # --------------------------------------------------------
    # Final spacing
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.09,
        right=0.98,
        top=0.90,
        bottom=0.24
    )


    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(
    width='295px'
)


slider_style = {
    'description_width': '115px'
}


display_selector = RadioButtons(
    options=[
        'FIR output noise vs N',
        'FIR noise contributions',
        'IIR output noise vs alpha',
        'FIR vs IIR',
        'Effect of word length K'
    ],
    value='FIR output noise vs N',
    description='Display:',
    style={'description_width': '65px'},
    layout=Layout(
        width='310px'
    )
)


K_slider = IntSlider(
    value=8,
    min=2,
    max=16,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


N_slider = IntSlider(
    value=12,
    min=2,
    max=64,
    step=1,
    description='FIR length N:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


alpha_slider = FloatSlider(
    value=0.85,
    min=-0.98,
    max=0.98,
    step=0.01,
    description='IIR alpha:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


# ------------------------------------------------------------
# Enable only relevant controls
# ------------------------------------------------------------

def update_control_states(change=None):

    mode = display_selector.value


    if mode == 'FIR output noise vs N':

        K_slider.disabled = False

        N_slider.disabled = False

        alpha_slider.disabled = True


    elif mode == 'FIR noise contributions':

        K_slider.disabled = False

        N_slider.disabled = False

        alpha_slider.disabled = True


    elif mode == 'IIR output noise vs alpha':

        K_slider.disabled = False

        N_slider.disabled = True

        alpha_slider.disabled = False


    elif mode == 'FIR vs IIR':

        K_slider.disabled = False

        N_slider.disabled = False

        alpha_slider.disabled = False


    else:

        K_slider.disabled = True

        N_slider.disabled = False

        alpha_slider.disabled = False


display_selector.observe(
    update_control_states,
    names='value'
)


# ------------------------------------------------------------
# Set initial control state
# ------------------------------------------------------------

update_control_states()


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_fir_iir_noise,
    display_mode=display_selector,
    K=K_slider,
    N=N_slider,
    alpha=alpha_slider
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls_box = VBox(
    [
        HTML("<div class='fi-title'>Controls</div>"),
        display_selector,
        K_slider,
        N_slider,
        alpha_slider
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final notebook layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)